In [ ]:
# Imports and folders


%matplotlib widget

import os
import glob
import pickle
import zipfile
import numpy as np
import cv2
import matplotlib.pyplot as plt

from scipy.spatial import cKDTree

os.makedirs("output", exist_ok=True)
os.makedirs("output/cache", exist_ok=True)
os.makedirs("output/ply", exist_ok=True)
os.makedirs("output/polygons", exist_ok=True)
os.makedirs("output/clicks", exist_ok=True)
os.makedirs("output/transforms", exist_ok=True)


In [ ]:
# Settings

DATA_DIR = "david"

scan_names = [
    "grab_0",
    "grab_1",
    "grab_2",
    "grab_3",
    "grab_4",
]

NBITS = 9

# Reconstruction
DECODE_THRESHOLD = 4
COLOR_THRESHOLD = 18

# Reconstruct again because image clicking needs pts2L and pts3 to match.
FORCE_RECONSTRUCT = True
FORCE_NEW_POLYGONS = False
FORCE_NEW_CLICKS = False

# Matching
# 1.0 cleaner, 1.5 denser.
CODE_RADIUS_TRIALS = [1.0, 1.5]
MIN_GOOD_MATCHES_TO_STOP = 50000

# Cleanup
MAX_EDGE_LENGTH = 4.0
SMOOTH_ITERS = 0
SMOOTH_ALPHA = 0.25

SINGLE_COMPONENT_RADIUS = 4.0
SINGLE_MIN_COMPONENT_SIZE = 1000

FINAL_COMPONENT_RADIUS = 5.0
FINAL_MIN_COMPONENT_SIZE = 3000

# Must be False for normal image alignment.
# If True, pts2L and pts3 will no longer match.
SINGLE_VOXEL_FUSE_ENABLED = False

# Alignment
ALIGN_PAIRS = [
    ("grab_1", "grab_0"),
    ("grab_2", "grab_1"),
    ("grab_3", "grab_2"),
    ("grab_4", "grab_3"),
]

NUM_ALIGN_POINTS = 6

# Use normal image clicking.
ALIGN_CLICK_MODE = "image"

USE_ICP_REFINEMENT = False

ICP_ITERS = 15
ICP_MAX_DIST = 3.0
ICP_SAMPLE = 40000

# Z cleanup
Z_FILTER_ENABLED = True
Z_PERCENTILE_LOW = 0.5
Z_PERCENTILE_HIGH = 99.5

# this is final voxel fuse
FINAL_VOXEL_CELL_SIZES = [0.18, 0.22, 0.26, 0.30, 0.35, 0.40]
TARGET_POINTS = 90000


In [ ]:
# Camera, triangulation, alignment, PLY saving

class Camera:
    def __init__(self, *args):
        if len(args) == 4:
            f, c, R, t = args
            self.fx = float(f)
            self.fy = float(f)
            self.f = float(f)
            self.c = np.asarray(c, dtype=float).reshape(2, 1)
            self.R = np.asarray(R, dtype=float).reshape(3, 3)
            self.t = np.asarray(t, dtype=float).reshape(3, 1)
            self.dist = None

        elif len(args) == 5:
            if np.isscalar(args[1]):
                fx, fy, c, R, t = args
                self.fx = float(fx)
                self.fy = float(fy)
                self.f = 0.5 * (self.fx + self.fy)
                self.c = np.asarray(c, dtype=float).reshape(2, 1)
                self.R = np.asarray(R, dtype=float).reshape(3, 3)
                self.t = np.asarray(t, dtype=float).reshape(3, 1)
                self.dist = None
            else:
                f, c, R, t, dist = args
                self.fx = float(f)
                self.fy = float(f)
                self.f = float(f)
                self.c = np.asarray(c, dtype=float).reshape(2, 1)
                self.R = np.asarray(R, dtype=float).reshape(3, 3)
                self.t = np.asarray(t, dtype=float).reshape(3, 1)
                self.dist = dist

        elif len(args) == 6:
            fx, fy, c, R, t, dist = args
            self.fx = float(fx)
            self.fy = float(fy)
            self.f = 0.5 * (self.fx + self.fy)
            self.c = np.asarray(c, dtype=float).reshape(2, 1)
            self.R = np.asarray(R, dtype=float).reshape(3, 3)
            self.t = np.asarray(t, dtype=float).reshape(3, 1)
            self.dist = dist

        else:
            raise TypeError("Invalid Camera constructor.")


def triangulate(pts2L, camL, pts2R, camR):
    assert pts2L.shape[0] == 2
    assert pts2R.shape[0] == 2
    assert pts2L.shape[1] == pts2R.shape[1]

    npts = pts2L.shape[1]

    qL = np.vstack((
        (pts2L[0] - camL.c[0, 0]) / camL.fx,
        (pts2L[1] - camL.c[1, 0]) / camL.fy,
        np.ones(npts)
    ))

    qR = np.vstack((
        (pts2R[0] - camR.c[0, 0]) / camR.fx,
        (pts2R[1] - camR.c[1, 0]) / camR.fy,
        np.ones(npts)
    ))

    xL = np.zeros((3, npts))
    xR = np.zeros((3, npts))

    b = (camR.t - camL.t).reshape(3)

    for i in range(npts):
        A = np.vstack((camL.R @ qL[:, i], -camR.R @ qR[:, i])).T
        z, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
        xL[:, i] = z[0] * qL[:, i]
        xR[:, i] = z[1] * qR[:, i]

    pts3L = camL.R @ xL + camL.t
    pts3R = camR.R @ xR + camR.t

    return 0.5 * (pts3L + pts3R)


def align_scans(ptsA, ptsB):
    """
    Compute R,t so:
        ptsB ≈ R @ ptsA + t
    """
    assert ptsA.shape == ptsB.shape
    assert ptsA.shape[0] == 3
    assert ptsA.shape[1] >= 3

    ca = np.mean(ptsA, axis=1, keepdims=True)
    cb = np.mean(ptsB, axis=1, keepdims=True)

    AA = ptsA - ca
    BB = ptsB - cb

    H = AA @ BB.T
    U, S, Vt = np.linalg.svd(H)

    R = Vt.T @ U.T

    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T

    t = cb - R @ ca

    return R, t


def apply_alignment(pts3, R, t):
    return R @ pts3 + t


def save_as_ply(filename, pts3, colors=None, faces=None):
    os.makedirs(os.path.dirname(filename), exist_ok=True)

    pts3 = np.asarray(pts3)
    assert pts3.shape[0] == 3
    npts = pts3.shape[1]

    if colors is not None:
        colors = np.asarray(colors)
        assert colors.shape == (3, npts)
        colors = np.clip(colors, 0, 255).astype(np.uint8)

    if faces is not None:
        faces = np.asarray(faces)
        if faces.size == 0:
            faces = None
        elif faces.shape[0] == 3 and faces.shape[1] != 3:
            faces = faces.T

    nfaces = 0 if faces is None else faces.shape[0]

    with open(filename, "w") as f:
        f.write("ply\n")
        f.write("format ascii 1.0\n")
        f.write(f"element vertex {npts}\n")
        f.write("property float x\n")
        f.write("property float y\n")
        f.write("property float z\n")

        if colors is not None:
            f.write("property uchar red\n")
            f.write("property uchar green\n")
            f.write("property uchar blue\n")

        if faces is not None:
            f.write(f"element face {nfaces}\n")
            f.write("property list uchar int vertex_indices\n")

        f.write("end_header\n")

        for i in range(npts):
            x, y, z = pts3[:, i]

            if colors is not None:
                r, g, b = colors[:, i]
                f.write(f"{x} {y} {z} {r} {g} {b}\n")
            else:
                f.write(f"{x} {y} {z}\n")

        if faces is not None:
            for tri in faces:
                f.write(f"3 {int(tri[0])} {int(tri[1])} {int(tri[2])}\n")



In [ ]:
# Load stereo calibration
# This cell loads the stereo camera calibration parameters.

def load_stereo_calibration(calib_file="stereo_calibration.pickle", calib_dir="calib"):


    # If the stereo calibration file has not been created yet,
    # run the stereo calibration script to generate it.
    if not os.path.exists(calib_file):
        print("stereo_calibration.pickle not found. Trying lib.stereo_calibrate...")

        # Try importing stereo_calibrate from the lib folder first.
        # If that fails, try importing it from the current directory.
        try:
            from lib.stereo_calibrate import stereo_calibrate
        except Exception:
            from stereo_calibrate import stereo_calibrate

        # Run stereo calibration using the checkerboard images in calib_dir.
        # The checkerboard has 8 by 6 inner corners, and each square is 2.8 units.
        stereo_calibrate(
            calib_dir=calib_dir,
            board_size=(8, 6),
            square_size=2.8,
            output_file=calib_file,
            show_corners=False
        )

    # Load the calibration data from the pickle file.
    with open(calib_file, "rb") as f:
        data = pickle.load(f)


    # K0 and K1 are the intrinsic matrices for the two cameras.
    # R_cv and T_cv describe the transformation between the cameras.
    if all(k in data for k in ["K0", "K1", "R_cv", "T_cv"]):
        K0 = data["K0"]
        K1 = data["K1"]

        # Distortion coefficients may or may not exist in the file.
        dist0 = data.get("dist0", None)
        dist1 = data.get("dist1", None)

        R_cv = data["R_cv"]
        T_cv = data["T_cv"]

        # Extract the principal point from the intrinsic matrices.
        c0 = np.array([[K0[0, 2]], [K0[1, 2]]])
        c1 = np.array([[K1[0, 2]], [K1[1, 2]]])

        # Use the left camera as the world coordinate reference.
        # This means the left camera has identity rotation and zero translation.
        R0 = np.eye(3)
        t0 = np.zeros((3, 1))

        # Convert OpenCV stereo calibration output into the Camera class format.
        # The right camera pose is expressed relative to the left camera.
        R1 = R_cv.T
        t1 = -R_cv.T @ T_cv

        # Create Camera objects for the left and right cameras.
        # The Camera class expects fx, fy, center, rotation, translation, and distortion.
        camL = Camera(K0[0, 0], K0[1, 1], c0, R0, t0, dist0)
        camR = Camera(K1[0, 0], K1[1, 1], c1, R1, t1, dist1)

        # Print useful information to make sure the calibration loaded correctly.
        print("Loaded stereo calibration:", calib_file)
        print("camL fx/fy:", camL.fx, camL.fy)
        print("camR fx/fy:", camR.fx, camR.fy)
        print("baseline:", np.linalg.norm(camR.t - camL.t))

        return camL, camR, data

    # Some earlier code saved the camera parameters using different key names.
    # This block keeps the notebook compatible with those older files.
    if all(k in data for k in [
        "camL_f", "camL_c", "camL_R", "camL_t",
        "camR_f", "camR_c", "camR_R", "camR_t"
    ]):
        camL = Camera(
            data["camL_f"],
            data["camL_c"],
            data["camL_R"],
            data["camL_t"],
            data.get("camL_dist", None)
        )

        camR = Camera(
            data["camR_f"],
            data["camR_c"],
            data["camR_R"],
            data["camR_t"],
            data.get("camR_dist", None)
        )

        print("Loaded older calibration format:", calib_file)

        return camL, camR, data

    # If neither format matches, the calibration file is not in a supported format.
    raise ValueError("Unknown stereo calibration format.")


# Load the stereo calibration and create the left and right Camera objects.
camL, camR, stereo_data = load_stereo_calibration(
    calib_file="stereo_calibration.pickle",
    calib_dir="calib"
)

print("Cell 4 complete.")

Loaded stereo calibration: stereo_calibration.pickle
camL fx/fy: 1411.4689008089185 1406.727224664993
camR fx/fy: 1415.4497730927183 1411.6819062827021
baseline: 11.707959849104604
Cell 4 complete.


In [ ]:
# File path helpers

def find_existing_file(candidates):
    for path in candidates:
        if os.path.exists(path):
            return path

    raise FileNotFoundError("Could not find any of:\n" + "\n".join(candidates))


def get_frame_path(scan_dir, cam_id, idx):
    return find_existing_file([
        os.path.join(scan_dir, f"frame_C{cam_id}_{idx:02d}_u.png"),
        os.path.join(scan_dir, f"frame_C{cam_id}_{idx:02d}.png"),
    ])


def get_color_path(scan_dir, cam_id, idx):
    return find_existing_file([
        os.path.join(scan_dir, f"color_C{cam_id}_{idx:02d}_u.png"),
        os.path.join(scan_dir, f"color_C{cam_id}_{idx:02d}.png"),
    ])


def get_scan_dir(scan_name):
    scan_dir = os.path.join(DATA_DIR, scan_name)

    if not os.path.isdir(scan_dir):
        raise FileNotFoundError(scan_dir)

    return scan_dir


for s in scan_names:
    sd = get_scan_dir(s)
    print(s)
    print("  frame:", get_frame_path(sd, 0, 0))
    print("  color:", get_color_path(sd, 0, 1))

print("Cell 5 complete.")

grab_0
  frame: david/grab_0/frame_C0_00_u.png
  color: david/grab_0/color_C0_01_u.png
grab_1
  frame: david/grab_1/frame_C0_00_u.png
  color: david/grab_1/color_C0_01_u.png
grab_2
  frame: david/grab_2/frame_C0_00_u.png
  color: david/grab_2/color_C0_01_u.png
grab_3
  frame: david/grab_3/frame_C0_00_u.png
  color: david/grab_3/color_C0_01_u.png
grab_4
  frame: david/grab_4/frame_C0_00_u.png
  color: david/grab_4/color_C0_01_u.png
Cell 5 complete.


In [ ]:
# Gray-code decoding
# This cell decodes the structured light pattern images into projector codes.


def decode_gray_from_paths(paths_a, paths_b, threshold):
    """
    Decode one set of Gray-code pattern images.

    paths_a and paths_b are paired pattern images.
    For each bit, one image is the normal pattern and the other is the inverse pattern.
    By comparing the brightness of the two images, we decide whether the bit is 0 or 1.

    The threshold is used to reject unreliable pixels.
    If the brightness difference between the two images is too small,
    that pixel is considered uncertain and removed from the mask.
    """

    # Read one image first so we know the image height and width.
    sample = cv2.imread(paths_a[0], cv2.IMREAD_GRAYSCALE)

    if sample is None:
        raise FileNotFoundError(paths_a[0])

    h, w = sample.shape
    nbits = len(paths_a)

    # gray_bits stores the decoded Gray-code bits for all pixels.
    # Shape is (number of bits, image height, image width).
    gray_bits = np.zeros((nbits, h, w), dtype=np.uint8)

    # Start with all pixels marked as valid.
    # Pixels will be removed from this mask if any bit is too uncertain.
    mask = np.ones((h, w), dtype=bool)

    # Decode each Gray-code bit by comparing a normal pattern image
    # with its inverse pattern image.
    for i, (pa, pb) in enumerate(zip(paths_a, paths_b)):
        img_a = cv2.imread(pa, cv2.IMREAD_GRAYSCALE)
        img_b = cv2.imread(pb, cv2.IMREAD_GRAYSCALE)

        if img_a is None:
            raise FileNotFoundError(pa)
        if img_b is None:
            raise FileNotFoundError(pb)

        # Convert to float so intensity differences are computed safely.
        img_a = img_a.astype(np.float32)
        img_b = img_b.astype(np.float32)

        # If img_a is brighter than img_b, the Gray-code bit is 1.
        # Otherwise, the bit is 0.
        gray_bits[i] = (img_a > img_b).astype(np.uint8)

        # Keep only pixels where the two images are different enough.
        # Small differences usually mean the bit is unreliable.
        mask &= (np.abs(img_a - img_b) > threshold)

    # Convert Gray code bits to normal binary bits.
    # The first binary bit is the same as the first Gray bit.
    binary_bits = np.zeros_like(gray_bits)
    binary_bits[0] = gray_bits[0]

    # Each next binary bit is computed by XORing the previous binary bit
    # with the current Gray bit.
    for i in range(1, nbits):
        binary_bits[i] = binary_bits[i - 1] ^ gray_bits[i]

    # Convert the binary bit image into an integer code image.
    # Each pixel now has one integer code value.
    code = np.zeros((h, w), dtype=np.int32)

    for i in range(nbits):
        code += binary_bits[i].astype(np.int32) * (2 ** (nbits - 1 - i))

    return code, mask


def decode_scan_codes(scan_dir, cam_id, threshold=6, nbits=9):
    """
    Decode both horizontal and vertical projector codes for one camera in one scan.

    The first 2 * nbits images are used for horizontal codes.
    The next 2 * nbits images are used for vertical codes.

    Returns:
        code: combined projector code for each pixel
        decode_mask: pixels that were reliable in both horizontal and vertical decoding
        H, V: decoded horizontal and vertical code images
        Hmask, Vmask: validity masks for H and V
    """

    # Build the file paths for the horizontal Gray-code image pairs.
    # For each bit, there is a normal image and an inverse image.
    h_paths_a = [get_frame_path(scan_dir, cam_id, 2 * i) for i in range(nbits)]
    h_paths_b = [get_frame_path(scan_dir, cam_id, 2 * i + 1) for i in range(nbits)]

    # Vertical patterns start after all horizontal pattern images.
    v_start = 2 * nbits

    # Build the file paths for the vertical Gray-code image pairs.
    v_paths_a = [get_frame_path(scan_dir, cam_id, v_start + 2 * i) for i in range(nbits)]
    v_paths_b = [get_frame_path(scan_dir, cam_id, v_start + 2 * i + 1) for i in range(nbits)]

    # Decode horizontal and vertical Gray-code patterns separately.
    H, Hmask = decode_gray_from_paths(h_paths_a, h_paths_b, threshold)
    V, Vmask = decode_gray_from_paths(v_paths_a, v_paths_b, threshold)

    # Combine horizontal and vertical codes into one unique projector code.
    # This makes it easier to match pixels between the two cameras.
    code = ((2 ** nbits) * H + V).astype(np.int64)

    # A pixel is valid only if both its horizontal and vertical codes are reliable.
    decode_mask = Hmask & Vmask

    return code, decode_mask, H, V, Hmask, Vmask


print("Cell 6 complete.")

In [ ]:
#  Object mask and polygon tools
# This cell provides helper functions for creating object masks.



def compute_object_mask(object_img, background_img, threshold=30):

    if object_img is None or background_img is None:
        raise ValueError("object_img and background_img cannot be None.")

    # Compute the absolute RGB difference between the object image and background image.
    diff = np.abs(object_img.astype(np.float32) - background_img.astype(np.float32))

    # Convert the RGB difference into a single grayscale difference value.
    diff_gray = np.mean(diff, axis=2)

    # Pixels with large enough difference are considered part of the object.
    return diff_gray > threshold


def clean_binary_mask(mask):


    kernel = np.ones((5, 5), np.uint8)

    # Fill small holes and connect small gaps.
    mask2 = cv2.morphologyEx(
        mask.astype(np.uint8),
        cv2.MORPH_CLOSE,
        kernel,
        iterations=1
    )

    # this code remove small isolated noisy regions.
    mask2 = cv2.morphologyEx(
        mask2,
        cv2.MORPH_OPEN,
        kernel,
        iterations=1
    )

    return mask2.astype(bool)


def polygon_to_mask(poly_xy, h, w):


    mask = np.zeros((h, w), dtype=np.uint8)

    # OpenCV expects polygon points in shape (N, 1, 2).
    pts = np.asarray(poly_xy, dtype=np.int32).reshape((-1, 1, 2))

    # Fill the inside of the polygon.
    cv2.fillPoly(mask, [pts], 1)

    return mask.astype(bool)


def click_polygon(img_rgb, title="Draw polygon"):


    # OpenCV displays images in BGR format, so convert RGB to BGR.
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    display = img_bgr.copy()

    # Store clicked polygon points here.
    points = []

    window_name = title
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

    def redraw():


        nonlocal display
        display = img_bgr.copy()

        # Draw each clicked point.
        for p in points:
            cv2.circle(display, tuple(p), 4, (0, 0, 255), -1)

        # Draw green lines between consecutive points.
        if len(points) >= 2:
            for i in range(len(points) - 1):
                cv2.line(
                    display,
                    tuple(points[i]),
                    tuple(points[i + 1]),
                    (0, 255, 0),
                    2
                )

        # If there are at least 3 points, draw a closing line back to the first point.
        if len(points) >= 3:
            cv2.line(
                display,
                tuple(points[-1]),
                tuple(points[0]),
                (255, 0, 0),
                1
            )

        # Show instructions in the image window.
        cv2.putText(
            display,
            "Left click object boundary. z=undo, c/enter/space=finish, q/esc=cancel",
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 255),
            2
        )

    def mouse_callback(event, x, y, flags, param):
        """
        Add a new polygon point whenever the user left-clicks.
        """

        if event == cv2.EVENT_LBUTTONDOWN:
            points.append([x, y])
            redraw()

    # Attach the mouse callback to the OpenCV window.
    cv2.setMouseCallback(window_name, mouse_callback)
    redraw()

    # Keep the window open until the user finishes or cancels.
    while True:
        cv2.imshow(window_name, display)
        key = cv2.waitKey(20) & 0xFF

        # Undo the last clicked point.
        if key == ord("z"):
            if points:
                points.pop()
                redraw()

        # Finish the polygon if it has at least 3 points.
        elif key in [ord("c"), 13, 32]:
            if len(points) >= 3:
                break
            print("Need at least 3 points.")

        # Cancel polygon drawing.
        elif key in [ord("q"), 27]:
            cv2.destroyWindow(window_name)
            raise RuntimeError("Polygon drawing cancelled.")

    cv2.destroyWindow(window_name)

    return np.asarray(points, dtype=np.float32)


def get_object_polygon_mask(scan_name, scan_dir, cam_id=0, force_new=False):


    # Each scan and camera gets its own saved polygon file.
    polygon_file = os.path.join(
        "output/polygons",
        f"object_polygon_{scan_name}_C{cam_id}.pickle"
    )

    # Load the object color image for this scan and camera.
    color_obj_path = get_color_path(scan_dir, cam_id=cam_id, idx=1)
    img_bgr = cv2.imread(color_obj_path, cv2.IMREAD_COLOR)

    if img_bgr is None:
        raise FileNotFoundError(color_obj_path)

    # Convert to RGB for easier display and consistency with matplotlib-style images.
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    # If the polygon already exists and we are not forcing a new one,
    # load the saved polygon points.
    if os.path.exists(polygon_file) and not force_new:
        with open(polygon_file, "rb") as f:
            data = pickle.load(f)

        poly_xy = data["poly_xy"]
        print(f"{scan_name} C{cam_id}: loaded polygon:", polygon_file)

    else:
        # Otherwise, manually draw a new polygon around the object.
        poly_xy = click_polygon(
            img_rgb,
            f"{scan_name} C{cam_id}: draw David polygon"
        )

        # Save the polygon so it can be reused later.
        with open(polygon_file, "wb") as f:
            pickle.dump({"poly_xy": poly_xy}, f)

        print(f"{scan_name} C{cam_id}: saved polygon:", polygon_file)

    # Convert polygon points into a binary mask image.
    mask = polygon_to_mask(poly_xy, h, w)

    return mask, poly_xy


print("Cell 7 complete.")

In [ ]:
# Result cleanup helpers
# This cell contains helper functions for cleaning reconstructed scan results.



def build_faces_from_image_grid(matchL, h, w):

    # index_map stores the new vertex index for each image pixel.
    # Pixels that were not reconstructed stay as -1.
    index_map = -np.ones(h * w, dtype=int)
    index_map[matchL] = np.arange(len(matchL))
    index_map = index_map.reshape(h, w)

    faces = []

    # Loop over each 2x2 pixel block in the image.
    # A 2x2 block can form two triangles if the required vertices exist.
    for y in range(h - 1):
        for x in range(w - 1):
            i00 = index_map[y, x]
            i10 = index_map[y, x + 1]
            i01 = index_map[y + 1, x]
            i11 = index_map[y + 1, x + 1]

            # First triangle in the 2x2 pixel cell.
            if i00 >= 0 and i10 >= 0 and i01 >= 0:
                faces.append([i00, i10, i01])

            # Second triangle in the 2x2 pixel cell.
            if i10 >= 0 and i11 >= 0 and i01 >= 0:
                faces.append([i10, i11, i01])

    return np.array(faces, dtype=int)


def filter_result_by_mask(result, keep):

    keep = np.asarray(keep).astype(bool)

    # old_to_new maps old vertex indices to new vertex indices.
    # Removed vertices get index -1.
    old_to_new = -np.ones(len(keep), dtype=int)
    old_to_new[keep] = np.arange(np.sum(keep))

    result2 = result.copy()

    # Filter point-based arrays.
    # These arrays all have shape (dimension, number of points).
    for key in ["pts3", "colors", "pts2L", "pts2R"]:
        if key in result2 and result2[key] is not None:
            result2[key] = result2[key][:, keep]

    # Filter match arrays.
    # These arrays have one value per reconstructed point.
    for key in ["matchL", "matchR"]:
        if key in result2 and result2[key] is not None:
            result2[key] = result2[key][keep]

    faces = result2.get("faces", None)

    # If faces exist, only keep triangles where all three vertices are kept.
    # Then remap old vertex indices to the new filtered vertex indices.
    if faces is not None and len(faces) > 0:
        faces = np.asarray(faces)

        # Some code may store faces as shape (3, F), so convert to (F, 3).
        if faces.shape[0] == 3 and faces.shape[1] != 3:
            faces = faces.T

        face_keep = keep[faces].all(axis=1)
        result2["faces"] = old_to_new[faces[face_keep]].astype(int)

    return result2


def remove_nonfinite_points(result):

    pts = result["pts3"]

    # Keep only points where x, y, and z are all finite numbers.
    keep = np.isfinite(pts).all(axis=0)

    print("remove_nonfinite:", np.sum(keep), "/", len(keep))

    return filter_result_by_mask(result, keep)


def remove_percentile_outliers(result, low=0.5, high=99.5):

    pts = result["pts3"]
    keep = np.ones(pts.shape[1], dtype=bool)

    # Apply percentile filtering separately on x, y, and z.
    for axis in range(3):
        lo, hi = np.percentile(pts[axis], [low, high])
        keep &= (pts[axis] >= lo) & (pts[axis] <= hi)

    print("remove_percentile_outliers:", np.sum(keep), "/", len(keep))

    return filter_result_by_mask(result, keep)


def remove_z_percentile_outliers(result, low=0.5, high=99.5):

    pts = result["pts3"]

    # Compute the valid z range based on percentiles.
    lo, hi = np.percentile(pts[2], [low, high])

    # Keep only points inside this z range.
    keep = (pts[2] >= lo) & (pts[2] <= hi)

    print("remove_z_percentile_outliers:", np.sum(keep), "/", len(keep))
    print("z range kept:", lo, hi)

    return filter_result_by_mask(result, keep)


def cleanup_long_edges(result, max_edge_length=3.0):

    faces = result.get("faces", None)

    # If this result has no faces, there is nothing to clean.
    if faces is None or len(faces) == 0:
        return result

    pts = result["pts3"]
    faces = np.asarray(faces)

    # Make sure faces are stored as (number_of_faces, 3).
    if faces.shape[0] == 3 and faces.shape[1] != 3:
        faces = faces.T

    kept = []

    # Check each triangle and remove it if any edge is too long.
    for tri in faces:
        i, j, k = map(int, tri)

        e01 = np.linalg.norm(pts[:, i] - pts[:, j])
        e12 = np.linalg.norm(pts[:, j] - pts[:, k])
        e20 = np.linalg.norm(pts[:, k] - pts[:, i])

        if max(e01, e12, e20) <= max_edge_length:
            kept.append([i, j, k])

    result2 = result.copy()
    result2["faces"] = np.array(kept, dtype=int)

    print("cleanup_long_edges faces:", len(result2["faces"]))

    return result2


def keep_largest_component(result, radius=4.0, min_component_size=1000):

    pts = result["pts3"].T
    n = pts.shape[0]

    if n == 0:
        return result

    print(f"KDTree component cleanup: n={n}, radius={radius}")

    # KDTree makes it fast to find nearby points.
    tree = cKDTree(pts)

    visited = np.zeros(n, dtype=bool)
    labels = -np.ones(n, dtype=int)
    comp_sizes = []

    comp_id = 0

    # Run a graph search over the point cloud.
    # Each unvisited point starts a new connected component.
    for i in range(n):
        if visited[i]:
            continue

        stack = [i]
        visited[i] = True
        labels[i] = comp_id
        size = 0

        while stack:
            v = stack.pop()
            size += 1

            # Find all points within the given radius.
            # These are treated as connected neighbors.
            for u in tree.query_ball_point(pts[v], radius):
                if not visited[u]:
                    visited[u] = True
                    labels[u] = comp_id
                    stack.append(u)

        comp_sizes.append(size)
        comp_id += 1

    comp_sizes = np.asarray(comp_sizes)

    if len(comp_sizes) == 0:
        return result

    # Find the largest connected component.
    best = int(np.argmax(comp_sizes))
    keep = labels == best

    print("components:", len(comp_sizes))
    print("largest component:", np.sum(keep), "/", n)

    # Keep only the largest component and remove all smaller floating pieces.
    return filter_result_by_mask(result, keep)


print("Cell 8 complete.")

In [ ]:
#  Polygon-first reconstruction with tolerant projector-code matching

def match_projector_codes_exact_preserve_left_order(codeL, codeR, valid_mask_L, valid_mask_R):
    """
    Exact backup:
        left projector code == right projector code
    """

    CL_valid = np.where(valid_mask_L.ravel(), codeL.ravel(), -1)
    CR_valid = np.where(valid_mask_R.ravel(), codeR.ravel(), -2)

    valid_left_idx = np.where(CL_valid >= 0)[0]
    valid_right_idx = np.where(CR_valid >= 0)[0]

    right_code_to_idx = {}

    for idx in valid_right_idx:
        code = int(CR_valid[idx])
        if code not in right_code_to_idx:
            right_code_to_idx[code] = idx

    matchL_list = []
    matchR_list = []

    for idxL in valid_left_idx:
        code = int(CL_valid[idxL])

        if code in right_code_to_idx:
            matchL_list.append(idxL)
            matchR_list.append(right_code_to_idx[code])

    return np.array(matchL_list, dtype=int), np.array(matchR_list, dtype=int)


def match_projector_codes_tolerant_preserve_left_order(
    H_L,
    V_L,
    H_R,
    V_R,
    valid_mask_L,
    valid_mask_R,
    code_radius=1.0
):
    """
    Tolerant matching:
    Match left/right pixels by nearest neighbor in projector-code space (H, V).
    """

    flat_valid_L = valid_mask_L.ravel()
    flat_valid_R = valid_mask_R.ravel()

    idxL_all = np.where(flat_valid_L)[0]
    idxR_all = np.where(flat_valid_R)[0]

    if len(idxL_all) == 0 or len(idxR_all) == 0:
        return np.array([], dtype=int), np.array([], dtype=int), np.array([])

    HL_flat = H_L.ravel()
    VL_flat = V_L.ravel()
    HR_flat = H_R.ravel()
    VR_flat = V_R.ravel()

    projL = np.vstack([
        HL_flat[idxL_all],
        VL_flat[idxL_all]
    ]).T.astype(np.float64)

    projR = np.vstack([
        HR_flat[idxR_all],
        VR_flat[idxR_all]
    ]).T.astype(np.float64)

    treeR = cKDTree(projR)

    dists, nn = treeR.query(projL, k=1)

    keep = dists <= code_radius

    matchL = idxL_all[keep]
    matchR = idxR_all[nn[keep]]

    return matchL.astype(int), matchR.astype(int), dists[keep]


def clean_binary_mask(mask):
    kernel = np.ones((3, 3), np.uint8)

    mask2 = cv2.morphologyEx(
        mask.astype(np.uint8),
        cv2.MORPH_CLOSE,
        kernel,
        iterations=1
    )

    mask2 = cv2.morphologyEx(
        mask2,
        cv2.MORPH_OPEN,
        kernel,
        iterations=1
    )

    return mask2.astype(bool)


def remove_duplicate_left_pixels(matchL, matchR, code_dists=None):
    """
    Keep only one match for each left pixel.
    """

    if len(matchL) == 0:
        return matchL, matchR, code_dists

    _, unique_idx = np.unique(matchL, return_index=True)
    unique_idx = np.sort(unique_idx)

    matchL2 = matchL[unique_idx]
    matchR2 = matchR[unique_idx]

    if code_dists is None:
        return matchL2, matchR2, None

    return matchL2, matchR2, code_dists[unique_idx]


def voxel_fuse_result_simple(result, cell_size=0.22):
    """
    Voxel fuse one point cloud.
    This reduces local density and makes point distribution more uniform.
    Faces are removed because voxel fusion changes vertex order.
    """

    pts = result["pts3"].T
    colors = result["colors"].T

    if pts.shape[0] == 0:
        return result

    voxel_keys = np.floor(pts / cell_size).astype(np.int64)

    voxel_dict = {}

    for i, key in enumerate(map(tuple, voxel_keys)):
        if key not in voxel_dict:
            voxel_dict[key] = []
        voxel_dict[key].append(i)

    fused_pts = []
    fused_colors = []

    for ids in voxel_dict.values():
        ids = np.asarray(ids)
        fused_pts.append(np.mean(pts[ids], axis=0))
        fused_colors.append(np.mean(colors[ids], axis=0))

    fused_pts = np.asarray(fused_pts).T
    fused_colors = np.asarray(fused_colors).T

    result2 = result.copy()
    result2["pts3"] = fused_pts
    result2["colors"] = fused_colors
    result2["faces"] = None

    print("Voxel fuse single scan:")
    print("  input points :", pts.shape[0])
    print("  output points:", fused_pts.shape[1])
    print("  cell_size    :", cell_size)

    return result2


def reconstruct_one_scan(scan_name, scan_dir):
    cache_file = os.path.join("output/cache", f"scan_{scan_name}_clean.pickle")

    if os.path.exists(cache_file) and not FORCE_RECONSTRUCT and not FORCE_NEW_POLYGONS:
        print(f"{scan_name}: loading cache:", cache_file)
        with open(cache_file, "rb") as f:
            return pickle.load(f)

    print("\n===================================")
    print("Reconstructing:", scan_name)
    print("===================================")

    colorL_bg_path = get_color_path(scan_dir, 0, 0)
    colorL_obj_path = get_color_path(scan_dir, 0, 1)

    colorR_bg_path = get_color_path(scan_dir, 1, 0)
    colorR_obj_path = get_color_path(scan_dir, 1, 1)

    colorL_bg_bgr = cv2.imread(colorL_bg_path, cv2.IMREAD_COLOR)
    colorL_obj_bgr = cv2.imread(colorL_obj_path, cv2.IMREAD_COLOR)

    colorR_bg_bgr = cv2.imread(colorR_bg_path, cv2.IMREAD_COLOR)
    colorR_obj_bgr = cv2.imread(colorR_obj_path, cv2.IMREAD_COLOR)

    if colorL_bg_bgr is None:
        raise FileNotFoundError(colorL_bg_path)
    if colorL_obj_bgr is None:
        raise FileNotFoundError(colorL_obj_path)
    if colorR_bg_bgr is None:
        raise FileNotFoundError(colorR_bg_path)
    if colorR_obj_bgr is None:
        raise FileNotFoundError(colorR_obj_path)

    h, w = colorL_obj_bgr.shape[:2]

    print("Using color files:")
    print("  C0 bg :", colorL_bg_path)
    print("  C0 obj:", colorL_obj_path)
    print("  C1 bg :", colorR_bg_path)
    print("  C1 obj:", colorR_obj_path)

    # Important: draw polygon on BOTH cameras.
    polygon_mask_L, poly_xy_L = get_object_polygon_mask(
        scan_name,
        scan_dir,
        cam_id=0,
        force_new=FORCE_NEW_POLYGONS
    )

    polygon_mask_R, poly_xy_R = get_object_polygon_mask(
        scan_name,
        scan_dir,
        cam_id=1,
        force_new=FORCE_NEW_POLYGONS
    )

    threshold_trials = [
        (DECODE_THRESHOLD, COLOR_THRESHOLD),
        (2, 15),
        (3, 15),
        (4, 18),
        (6, 20),
        (8, 25),
        (10, 30),
    ]

    code_radius_trials = CODE_RADIUS_TRIALS

    best_data = None

    for decode_thr, color_thr in threshold_trials:
        print("\n--- Trying thresholds ---")
        print("decode_thr =", decode_thr, "color_thr =", color_thr)

        codeL, decode_mask_L, H_L, V_L, Hmask_L, Vmask_L = decode_scan_codes(
            scan_dir,
            cam_id=0,
            threshold=decode_thr,
            nbits=NBITS
        )

        codeR, decode_mask_R, H_R, V_R, Hmask_R, Vmask_R = decode_scan_codes(
            scan_dir,
            cam_id=1,
            threshold=decode_thr,
            nbits=NBITS
        )

        objL_normal = clean_binary_mask(
            compute_object_mask(colorL_obj_bgr, colorL_bg_bgr, threshold=color_thr)
        )

        objR_normal = clean_binary_mask(
            compute_object_mask(colorR_obj_bgr, colorR_bg_bgr, threshold=color_thr)
        )

        # Some grabs have reversed 00/01 behavior.
        # Try reversed masks too.
        objL_reversed = clean_binary_mask(
            compute_object_mask(colorL_bg_bgr, colorL_obj_bgr, threshold=color_thr)
        )

        objR_reversed = clean_binary_mask(
            compute_object_mask(colorR_bg_bgr, colorR_obj_bgr, threshold=color_thr)
        )

        valid_L_poly = decode_mask_L & polygon_mask_L
        valid_R_poly = decode_mask_R & polygon_mask_R

        print("decode L :", np.sum(decode_mask_L))
        print("decode R :", np.sum(decode_mask_R))
        print("polygon L:", np.sum(polygon_mask_L))
        print("polygon R:", np.sum(polygon_mask_R))
        print("objL normal :", np.sum(objL_normal), "objR normal :", np.sum(objR_normal))
        print("objL reverse:", np.sum(objL_reversed), "objR reverse:", np.sum(objR_reversed))
        print("valid poly L:", np.sum(valid_L_poly))
        print("valid poly R:", np.sum(valid_R_poly))

        mask_modes = []

        # Normal color both cameras.
        if np.sum(objL_normal) > 100 and np.sum(objR_normal) > 100:
            mask_modes.append((
                "polygon_plus_normal_color_both",
                decode_mask_L & polygon_mask_L & objL_normal,
                decode_mask_R & polygon_mask_R & objR_normal,
                objL_normal,
                objR_normal,
                1.10
            ))

        # Reversed color both cameras.
        if np.sum(objL_reversed) > 100 and np.sum(objR_reversed) > 100:
            mask_modes.append((
                "polygon_plus_reversed_color_both",
                decode_mask_L & polygon_mask_L & objL_reversed,
                decode_mask_R & polygon_mask_R & objR_reversed,
                objL_reversed,
                objR_reversed,
                1.00
            ))

        # Left normal only.
        if np.sum(objL_normal) > 100:
            mask_modes.append((
                "polygon_plus_left_normal_only",
                decode_mask_L & polygon_mask_L & objL_normal,
                decode_mask_R & polygon_mask_R,
                objL_normal,
                np.ones_like(decode_mask_R, dtype=bool),
                0.90
            ))

        # Right normal only.
        if np.sum(objR_normal) > 100:
            mask_modes.append((
                "polygon_plus_right_normal_only",
                decode_mask_L & polygon_mask_L,
                decode_mask_R & polygon_mask_R & objR_normal,
                np.ones_like(decode_mask_L, dtype=bool),
                objR_normal,
                0.90
            ))

        # Left reversed only.
        if np.sum(objL_reversed) > 100:
            mask_modes.append((
                "polygon_plus_left_reversed_only",
                decode_mask_L & polygon_mask_L & objL_reversed,
                decode_mask_R & polygon_mask_R,
                objL_reversed,
                np.ones_like(decode_mask_R, dtype=bool),
                0.85
            ))

        # Right reversed only.
        if np.sum(objR_reversed) > 100:
            mask_modes.append((
                "polygon_plus_right_reversed_only",
                decode_mask_L & polygon_mask_L,
                decode_mask_R & polygon_mask_R & objR_reversed,
                np.ones_like(decode_mask_L, dtype=bool),
                objR_reversed,
                0.85
            ))

        # This is important because your color masks often fail.
        # Still restricted by BOTH-camera polygons.
        mask_modes.append((
            "polygon_decode_fallback_both_cameras",
            valid_L_poly,
            valid_R_poly,
            np.ones_like(decode_mask_L, dtype=bool),
            np.ones_like(decode_mask_R, dtype=bool),
            0.75
        ))

        for mode_name, valid_mask_L, valid_mask_R, object_mask_L_used, object_mask_R_used, mode_weight in mask_modes:
            if np.sum(valid_mask_L) < 500 or np.sum(valid_mask_R) < 500:
                print(f"Skipping {mode_name}: mask too small")
                continue

            # First try exact match.
            matchL_exact, matchR_exact = match_projector_codes_exact_preserve_left_order(
                codeL,
                codeR,
                valid_mask_L,
                valid_mask_R
            )

            matchL_exact, matchR_exact, _ = remove_duplicate_left_pixels(
                matchL_exact,
                matchR_exact,
                None
            )

            print(
                f"{mode_name + '_exact':42s} "
                f"L={np.sum(valid_mask_L):7d} "
                f"R={np.sum(valid_mask_R):7d} "
                f"matches={len(matchL_exact):7d}"
            )

            if len(matchL_exact) > 0:
                score = len(matchL_exact) * mode_weight * 0.95

                candidate = {
                    "score": score,
                    "mode_name": mode_name + "_exact",
                    "decode_thr": decode_thr,
                    "color_thr": color_thr,
                    "code_radius": 0.0,
                    "mean_code_dist": 0.0,
                    "decode_mask_L": decode_mask_L,
                    "decode_mask_R": decode_mask_R,
                    "object_mask_L": object_mask_L_used,
                    "object_mask_R": object_mask_R_used,
                    "valid_mask_L": valid_mask_L,
                    "valid_mask_R": valid_mask_R,
                    "matchL": matchL_exact,
                    "matchR": matchR_exact,
                }

                if best_data is None or candidate["score"] > best_data["score"]:
                    best_data = candidate

            # Then try tolerant match.
            for code_radius in code_radius_trials:
                matchL, matchR, code_dists = match_projector_codes_tolerant_preserve_left_order(
                    H_L,
                    V_L,
                    H_R,
                    V_R,
                    valid_mask_L,
                    valid_mask_R,
                    code_radius=code_radius
                )

                matchL, matchR, code_dists = remove_duplicate_left_pixels(
                    matchL,
                    matchR,
                    code_dists
                )

                mean_code_dist = np.mean(code_dists) if len(code_dists) > 0 else -1

                print(
                    f"{mode_name:42s} "
                    f"radius={code_radius:3.1f} "
                    f"L={np.sum(valid_mask_L):7d} "
                    f"R={np.sum(valid_mask_R):7d} "
                    f"matches={len(matchL):7d} "
                    f"mean_code_dist={mean_code_dist:.3f}"
                )

                if len(matchL) == 0:
                    continue

                # Prefer radius 1.0. Radius 1.5 is allowed but slightly penalized.
                radius_penalty = 1.0
                if code_radius > 1.0:
                    radius_penalty = 0.88

                score = len(matchL) * mode_weight * radius_penalty

                candidate = {
                    "score": score,
                    "mode_name": mode_name,
                    "decode_thr": decode_thr,
                    "color_thr": color_thr,
                    "code_radius": code_radius,
                    "mean_code_dist": mean_code_dist,
                    "decode_mask_L": decode_mask_L,
                    "decode_mask_R": decode_mask_R,
                    "object_mask_L": object_mask_L_used,
                    "object_mask_R": object_mask_R_used,
                    "valid_mask_L": valid_mask_L,
                    "valid_mask_R": valid_mask_R,
                    "matchL": matchL,
                    "matchR": matchR,
                }

                if best_data is None or candidate["score"] > best_data["score"]:
                    best_data = candidate

        if best_data is not None and len(best_data["matchL"]) >= MIN_GOOD_MATCHES_TO_STOP:
            print("Enough matches. Stop threshold search.")
            break

    if best_data is None or len(best_data["matchL"]) == 0:
        raise RuntimeError(
            f"{scan_name}: zero matched points. "
            "Try redrawing polygons tighter or increasing CODE_RADIUS_TRIALS to [1.0, 1.5, 2.0]."
        )

    print("\nBest mode for", scan_name)
    print("mode          :", best_data["mode_name"])
    print("decode_thr    :", best_data["decode_thr"])
    print("color_thr     :", best_data["color_thr"])
    print("code_radius   :", best_data["code_radius"])
    print("mean_code_dist:", best_data["mean_code_dist"])
    print("matches       :", len(best_data["matchL"]))

    matchL = best_data["matchL"]
    matchR = best_data["matchR"]

    yy, xx = np.divmod(np.arange(h * w), w)

    pts2L = np.vstack((xx[matchL], yy[matchL])).astype(np.float64)
    pts2R = np.vstack((xx[matchR], yy[matchR])).astype(np.float64)

    pts3 = triangulate(pts2L, camL, pts2R, camR)

    colorL_obj_rgb = cv2.cvtColor(colorL_obj_bgr, cv2.COLOR_BGR2RGB)
    colors = colorL_obj_rgb[yy[matchL], xx[matchL], :].T

    faces = build_faces_from_image_grid(matchL, h, w)

    result = {
        "scan_name": scan_name,
        "scan_dir": scan_dir,

        "pts3": pts3,
        "pts2L": pts2L,
        "pts2R": pts2R,

        "colors": colors,
        "faces": faces,

        "matchL": matchL,
        "matchR": matchR,

        "decode_mask_L": best_data["decode_mask_L"],
        "decode_mask_R": best_data["decode_mask_R"],

        "object_mask_L": best_data["object_mask_L"],
        "object_mask_R": best_data["object_mask_R"],

        "valid_mask_L": best_data["valid_mask_L"],
        "valid_mask_R": best_data["valid_mask_R"],

        "decode_threshold_used": best_data["decode_thr"],
        "color_threshold_used": best_data["color_thr"],
        "code_radius_used": best_data["code_radius"],
        "mask_mode_used": best_data["mode_name"],

        "polygon_xy_L": poly_xy_L,
        "polygon_xy_R": poly_xy_R,
        "polygon_mask_L": polygon_mask_L,
        "polygon_mask_R": polygon_mask_R,
    }

    print("Raw pts:", result["pts3"].shape[1])
    print("Raw faces:", 0 if result["faces"] is None else len(result["faces"]))

    # Cleanup
    result = remove_nonfinite_points(result)

    if Z_FILTER_ENABLED:
        result = remove_z_percentile_outliers(
            result,
            low=Z_PERCENTILE_LOW,
            high=Z_PERCENTILE_HIGH
        )

    result = remove_percentile_outliers(
        result,
        low=0.5,
        high=99.5
    )

    result = cleanup_long_edges(
        result,
        max_edge_length=MAX_EDGE_LENGTH
    )

    result = keep_largest_component(
        result,
        radius=SINGLE_COMPONENT_RADIUS,
        min_component_size=SINGLE_MIN_COMPONENT_SIZE
    )

    if SINGLE_VOXEL_FUSE_ENABLED:
        result = voxel_fuse_result_simple(
            result,
            cell_size=SINGLE_VOXEL_CELL_SIZE
        )

    print("Clean pts:", result["pts3"].shape[1])
    print("Clean faces:", 0 if result["faces"] is None else len(result["faces"]))

    clean_path = os.path.join(
        "output/ply",
        f"{scan_name}_clean.ply"
    )

    save_as_ply(
        clean_path,
        result["pts3"],
        colors=result["colors"],
        faces=result["faces"]
    )

    with open(cache_file, "wb") as f:
        pickle.dump(result, f)

    print("Saved:", clean_path)
    print("Saved cache:", cache_file)

    return result


print("Cell 9 complete: polygon-first tolerant reconstruction.")

Cell 9 complete: polygon-first tolerant reconstruction.


In [ ]:
# Cell 10: Reconstruct all scans

all_results_raw = {}

for scan_name in scan_names:
    scan_dir = get_scan_dir(scan_name)
    all_results_raw[scan_name] = reconstruct_one_scan(scan_name, scan_dir)

print("\nAll scans reconstructed:")
for name in scan_names:
    result = all_results_raw[name]
    print(
        name,
        "points:",
        result["pts3"].shape[1],
        "faces:",
        0 if result.get("faces", None) is None else len(result["faces"]),
        "mode:",
        result.get("mask_mode_used", "unknown"),
        "radius:",
        result.get("code_radius_used", "unknown")
    )

print("Cell 10 complete.")

In [ ]:
# Visualize single scans before alignment
# This cell displays each cleaned single scan before alignment.



def set_axes_equal(ax):

    # Get the current axis limits.
    x_limits = ax.get_xlim3d()
    y_limits = ax.get_ylim3d()
    z_limits = ax.get_zlim3d()

    # Compute the size of each axis range.
    x_range = abs(x_limits[1] - x_limits[0])
    y_range = abs(y_limits[1] - y_limits[0])
    z_range = abs(z_limits[1] - z_limits[0])

    # Compute the center of each axis.
    x_mid = np.mean(x_limits)
    y_mid = np.mean(y_limits)
    z_mid = np.mean(z_limits)

    # Use the largest range so all axes share the same scale.
    radius = 0.5 * max([x_range, y_range, z_range])

    # Reset the axis limits with equal scale.
    ax.set_xlim3d([x_mid - radius, x_mid + radius])
    ax.set_ylim3d([y_mid - radius, y_mid + radius])
    ax.set_zlim3d([z_mid - radius, z_mid + radius])


# Loop through every scan and show its cleaned point cloud.
for scan_name in scan_names:
    result = all_results_raw[scan_name]


    pts = result["pts3"]

    # Convert colors from 0-255 range to 0-1 range for matplotlib.
    colors = result["colors"].T / 255.0

    # Plot at most about 40,000 points so the visualization is not too slow.
    # This only affects display, not the actual reconstruction data.
    step = max(1, pts.shape[1] // 40000)

    fig = plt.figure(figsize=(8, 7))
    ax = fig.add_subplot(111, projection="3d")

    # Draw a subsampled colored point cloud.
    ax.scatter(
        pts[0, ::step],
        pts[1, ::step],
        pts[2, ::step],
        s=1,
        c=colors[::step]
    )

    ax.set_title(f"{scan_name} clean single scan")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")

    # Make sure the object is not visually distorted by unequal axis scaling.
    set_axes_equal(ax)

    plt.show()

print("Cell 11 complete.")

In [ ]:
# Robust side-by-side normal image clicking alignment helpers

import os
import glob
import pickle
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

os.makedirs("output/clicks", exist_ok=True)
os.makedirs("output/transforms", exist_ok=True)
os.makedirs("output/ply", exist_ok=True)


def estimate_rigid_transform(source_pts, target_pts):
    """
    Estimate rigid transform R, t such that:

        target_pts ≈ R @ source_pts + t

    source_pts: shape (3, N)
    target_pts: shape (3, N)
    """

    if source_pts.shape != target_pts.shape:
        raise ValueError(
            f"source_pts and target_pts must have same shape, "
            f"got {source_pts.shape} and {target_pts.shape}"
        )

    if source_pts.shape[0] != 3:
        raise ValueError("points must have shape (3, N).")

    if source_pts.shape[1] < 3:
        raise ValueError("Need at least 3 corresponding points.")

    source_center = np.mean(source_pts, axis=1, keepdims=True)
    target_center = np.mean(target_pts, axis=1, keepdims=True)

    source_zero = source_pts - source_center
    target_zero = target_pts - target_center

    H = source_zero @ target_zero.T

    U, S, Vt = np.linalg.svd(H)

    R = Vt.T @ U.T

    # Fix reflection
    if np.linalg.det(R) < 0:
        print("Reflection detected. Fixing rotation.")
        Vt[-1, :] *= -1
        R = Vt.T @ U.T

    t = target_center - R @ source_center

    return R, t


def load_left_color_image(result):
    """
    Load normal object color image from left camera C0.
    Usually this is color_C0_01.
    """

    scan_dir = result["scan_dir"]
    img_path = get_color_path(scan_dir, 0, 1)

    img_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)

    if img_bgr is None:
        raise FileNotFoundError(img_path)

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    return img_rgb, img_path


def check_pts2_pts3_match(result, name):
    """
    Normal image clicking requires pts2L and pts3 to be aligned.

    pts2L[:, i] should correspond to pts3[:, i].

    Therefore, do NOT use single-scan voxel fuse before alignment.
    """

    if "pts2L" not in result:
        raise RuntimeError(
            f"{name}: result does not contain pts2L. "
            "Your reconstruction result must save pts2L for image clicking."
        )

    if "pts3" not in result:
        raise RuntimeError(
            f"{name}: result does not contain pts3."
        )

    pts2L = result["pts2L"].T
    pts3 = result["pts3"]

    if pts2L.shape[0] != pts3.shape[1]:
        raise RuntimeError(
            f"{name}: pts2L and pts3 do not match.\n"
            f"pts2L count = {pts2L.shape[0]}\n"
            f"pts3 count  = {pts3.shape[1]}\n\n"
            "Normal image clicking cannot safely continue.\n"
            "Fix:\n"
            "1. Set SINGLE_VOXEL_FUSE_ENABLED = False in Cell 2.\n"
            "2. Set FORCE_RECONSTRUCT = True.\n"
            "3. Rerun Cell 9 and Cell 10.\n"
            "4. Then rerun Cell 12 and Cell 13."
        )


def nearest_3d_points_from_image_clicks(result, clicks, max_pixel_dist=8.0):
    """
    Convert clicked image pixels to 3D points.

    This version also checks how far the clicked pixel is from
    the nearest reconstructed 2D point. If the distance is large,
    that click is unreliable for alignment.
    """

    pts2L = result["pts2L"].T
    pts3 = result["pts3"]

    picked_ids = []
    picked_pts3 = []
    pixel_dists = []

    for i, click_xy in enumerate(clicks):
        d = np.linalg.norm(pts2L - click_xy[None, :], axis=1)
        nearest_id = int(np.argmin(d))
        nearest_dist = float(d[nearest_id])

        picked_ids.append(nearest_id)
        picked_pts3.append(pts3[:, nearest_id])
        pixel_dists.append(nearest_dist)

        print(
            f"click {i + 1}: "
            f"clicked=({click_xy[0]:.1f}, {click_xy[1]:.1f}), "
            f"nearest pts2L=({pts2L[nearest_id, 0]:.1f}, {pts2L[nearest_id, 1]:.1f}), "
            f"pixel_dist={nearest_dist:.2f}"
        )

        if nearest_dist > max_pixel_dist:
            print(
                f"WARNING: click {i + 1} is far from reconstructed points. "
                f"pixel_dist={nearest_dist:.2f} > {max_pixel_dist}. "
                "This point may hurt alignment."
            )

    picked_ids = np.asarray(picked_ids, dtype=int)
    picked_pts3 = np.asarray(picked_pts3).T
    pixel_dists = np.asarray(pixel_dists)

    print("pixel distances:", pixel_dists)
    print("mean pixel distance:", np.mean(pixel_dists))
    print("max pixel distance:", np.max(pixel_dists))

    return picked_pts3, picked_ids


def click_correspondences_side_by_side(
    target_result,
    source_result,
    target_name,
    source_name,
    n_points=6
):
    """
    Robust side-by-side image clicking.

    LEFT  = TARGET image
    RIGHT = SOURCE image

    Click order:
        1. Click n_points on LEFT target image.
        2. Click n_points on RIGHT source image.

    This version avoids plt.show(block=True), because block=True can get stuck
    in Jupyter / VSCode after the first pair.
    """

    check_pts2_pts3_match(target_result, target_name)
    check_pts2_pts3_match(source_result, source_name)

    target_img, target_img_path = load_left_color_image(target_result)
    source_img, source_img_path = load_left_color_image(source_result)

    target_clicks = []
    source_clicks = []

    finished = {"done": False}

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    ax_target = axes[0]
    ax_source = axes[1]

    ax_target.imshow(target_img)
    ax_target.set_title(
        f"TARGET {target_name}\n"
        f"Click points 1-{n_points} here first"
    )
    ax_target.axis("image")

    ax_source.imshow(source_img)
    ax_source.set_title(
        f"SOURCE {source_name}\n"
        f"Then click matching points 1-{n_points} here"
    )
    ax_source.axis("image")

    status_text = fig.suptitle(
        f"Alignment: {source_name} -> {target_name}\n"
        f"LEFT target: click point 1/{n_points}",
        fontsize=14
    )

    plt.tight_layout()

    print("\n======================================")
    print(f"Alignment: {source_name} -> {target_name}")
    print("LEFT  image = TARGET:", target_name)
    print("RIGHT image = SOURCE:", source_name)
    print(f"Click {n_points} points on LEFT first.")
    print(f"Then click {n_points} corresponding points on RIGHT.")
    print("Point order must match.")
    print("======================================")

    def update_status():
        if len(target_clicks) < n_points:
            status_text.set_text(
                f"Alignment: {source_name} -> {target_name}\n"
                f"LEFT target: click point {len(target_clicks) + 1}/{n_points}"
            )
        elif len(source_clicks) < n_points:
            status_text.set_text(
                f"Alignment: {source_name} -> {target_name}\n"
                f"RIGHT source: click point {len(source_clicks) + 1}/{n_points}"
            )
        else:
            status_text.set_text(
                f"Alignment: {source_name} -> {target_name}\n"
                "Finished. Closing window..."
            )

        fig.canvas.draw_idle()

    def draw_click(ax, x, y, idx):
        ax.scatter(x, y, s=110, marker="x", color="red")
        ax.text(
            x + 5,
            y + 5,
            str(idx),
            color="red",
            fontsize=16,
            fontweight="bold"
        )
        fig.canvas.draw_idle()

    def on_click(event):
        if finished["done"]:
            return

        if event.inaxes not in [ax_target, ax_source]:
            print("Ignored click outside image.")
            return

        # Stage 1: click target on left
        if len(target_clicks) < n_points:
            if event.inaxes != ax_target:
                print("Please click on LEFT target image first.")
                return

            x = float(event.xdata)
            y = float(event.ydata)

            target_clicks.append([x, y])
            idx = len(target_clicks)

            draw_click(ax_target, x, y, idx)
            print(f"TARGET point {idx}: ({x:.1f}, {y:.1f})")

            update_status()
            return

        # Stage 2: click source on right
        if len(source_clicks) < n_points:
            if event.inaxes != ax_source:
                print("Now please click on RIGHT source image.")
                return

            x = float(event.xdata)
            y = float(event.ydata)

            source_clicks.append([x, y])
            idx = len(source_clicks)

            draw_click(ax_source, x, y, idx)
            print(f"SOURCE point {idx}: ({x:.1f}, {y:.1f})")

            update_status()

            if len(source_clicks) == n_points:
                finished["done"] = True

                debug_path = os.path.join(
                    "output/clicks",
                    f"click_visual_{source_name}_to_{target_name}.png"
                )

                fig.savefig(debug_path, dpi=150, bbox_inches="tight")
                print("Saved click visualization:", debug_path)
                print("Finished clicking this pair. Moving to next pair...")

            return

    cid = fig.canvas.mpl_connect("button_press_event", on_click)

    update_status()
    plt.show(block=False)

    # This loop keeps the window interactive but returns after enough clicks.
    while not finished["done"]:
        plt.pause(0.1)

        # If user manually closes the window before finishing
        if not plt.fignum_exists(fig.number):
            break

    try:
        fig.canvas.mpl_disconnect(cid)
    except Exception:
        pass

    if plt.fignum_exists(fig.number):
        plt.pause(0.3)
        plt.close(fig)

    target_clicks = np.asarray(target_clicks, dtype=np.float64)
    source_clicks = np.asarray(source_clicks, dtype=np.float64)

    if target_clicks.shape[0] != n_points:
        raise RuntimeError(
            f"Expected {n_points} target clicks, got {target_clicks.shape[0]}."
        )

    if source_clicks.shape[0] != n_points:
        raise RuntimeError(
            f"Expected {n_points} source clicks, got {source_clicks.shape[0]}."
        )

    target_pts, target_ids = nearest_3d_points_from_image_clicks(
        target_result,
        target_clicks
    )

    source_pts, source_ids = nearest_3d_points_from_image_clicks(
        source_result,
        source_clicks
    )

    print("\nFinished side-by-side clicks.")
    print("target clicked pixels:")
    print(target_clicks)
    print("source clicked pixels:")
    print(source_clicks)
    print("target nearest ids:", target_ids)
    print("source nearest ids:", source_ids)

    return (
        target_pts,
        source_pts,
        target_ids,
        source_ids,
        target_clicks,
        source_clicks
    )


def manual_alignment_from_clicks(
    target_result,
    source_result,
    target_name,
    source_name
):
    """
    Estimate transform source -> target using side-by-side normal image clicks.
    """

    print("\n==============================")
    print("Manual alignment")
    print(source_name, "->", target_name)
    print("==============================")

    click_file = os.path.join(
        "output/clicks",
        f"side_by_side_clicks_{source_name}_to_{target_name}.pickle"
    )

    if os.path.exists(click_file) and not FORCE_NEW_CLICKS:
        print("Loading saved clicks:", click_file)

        with open(click_file, "rb") as f:
            data = pickle.load(f)

        target_pts = data["target_pts"]
        source_pts = data["source_pts"]
        target_ids = data["target_ids"]
        source_ids = data["source_ids"]

    else:
        (
            target_pts,
            source_pts,
            target_ids,
            source_ids,
            target_clicks,
            source_clicks,
        ) = click_correspondences_side_by_side(
            target_result,
            source_result,
            target_name,
            source_name,
            n_points=NUM_ALIGN_POINTS
        )

        with open(click_file, "wb") as f:
            pickle.dump(
                {
                    "target_pts": target_pts,
                    "source_pts": source_pts,
                    "target_ids": target_ids,
                    "source_ids": source_ids,
                    "target_name": target_name,
                    "source_name": source_name,
                    "target_clicks": target_clicks,
                    "source_clicks": source_clicks,
                },
                f
            )

        print("Saved clicks:", click_file)

    R, t = estimate_rigid_transform(source_pts, target_pts)

    source_aligned = R @ source_pts + t
    errors = np.linalg.norm(source_aligned - target_pts, axis=0)

    print("\nManual click alignment errors:")
    print(errors)
    print("mean error:", np.mean(errors))
    print("max error:", np.max(errors))

    return R, t


def icp_refine_transform(
    source_pts,
    target_pts,
    R_init,
    t_init,
    max_iters=15,
    max_dist=3.0,
    sample=40000
):
    """
    Point-to-point ICP refinement.

    Inputs:
        source_pts: source point cloud, shape (3, N)
        target_pts: target point cloud, shape (3, M)
        R_init, t_init: initial source -> target transform

    Returns:
        R_total, t_total
    """

    if source_pts.shape[1] == 0 or target_pts.shape[1] == 0:
        print("ICP skipped: empty cloud.")
        return R_init, t_init

    rng = np.random.default_rng(1)

    if source_pts.shape[1] > sample:
        source_ids = rng.choice(source_pts.shape[1], size=sample, replace=False)
        source_use = source_pts[:, source_ids]
    else:
        source_use = source_pts

    if target_pts.shape[1] > sample:
        target_ids = rng.choice(target_pts.shape[1], size=sample, replace=False)
        target_use = target_pts[:, target_ids]
    else:
        target_use = target_pts

    tree = cKDTree(target_use.T)

    R_total = R_init.copy()
    t_total = t_init.copy()

    for it in range(max_iters):
        source_aligned = R_total @ source_use + t_total

        dists, nn = tree.query(source_aligned.T, k=1)

        keep = dists < max_dist

        if np.sum(keep) < 20:
            print("ICP stopped: too few close pairs.")
            break

        A = source_aligned[:, keep]
        B = target_use[:, nn[keep]]

        dR, dt = estimate_rigid_transform(A, B)

        R_total = dR @ R_total
        t_total = dR @ t_total + dt

        mean_dist = np.mean(dists[keep])

        print(
            f"ICP iter {it + 1:02d}: "
            f"kept={np.sum(keep)} "
            f"mean_dist={mean_dist:.4f}"
        )

    return R_total, t_total


print("Cell 12 complete: robust side-by-side normal image alignment.")

Cell 12 complete: robust side-by-side normal image alignment.


In [ ]:


%matplotlib tk

import matplotlib
print("Matplotlib backend:", matplotlib.get_backend())

Matplotlib backend: qtagg


In [ ]:
# Pairwise chain alignment using side-by-side normal image clicks
# This cell aligns all reconstructed scans into one shared coordinate system.


import os
import glob
import pickle
import numpy as np

# Create output folders for saved clicks, transforms, and aligned PLY files.
os.makedirs("output/clicks", exist_ok=True)
os.makedirs("output/transforms", exist_ok=True)
os.makedirs("output/ply", exist_ok=True)


# If FORCE_NEW_CLICKS is True, delete old click files and transform files.
# This is useful when the old manual clicks were bad and I want to re-click alignment points.
if FORCE_NEW_CLICKS:
    for path in glob.glob("output/clicks/*.pickle"):
        print("Deleting old click:", path)
        os.remove(path)

    for path in glob.glob("output/transforms/*.pickle"):
        print("Deleting old transform:", path)
        os.remove(path)


# Store all aligned scan results in this dictionary.
aligned_results = {}

# Use grab_0 as the base coordinate system.
# It does not need to be transformed.
base_name = "grab_0"
aligned_results[base_name] = all_results_raw[base_name]

# Save the base scan as an aligned scan so it can be inspected in MeshLab.
save_as_ply(
    os.path.join("output/ply", f"{base_name}_aligned.ply"),
    aligned_results[base_name]["pts3"],
    colors=aligned_results[base_name]["colors"],
    faces=aligned_results[base_name].get("faces", None)
)

# Store each pairwise transform for debugging or later inspection.
pair_transforms = {}


# Align each source scan to its target scan.
# The order of ALIGN_PAIRS matters because this is chain alignment.
# For example:
#   grab_1 -> grab_0
#   grab_2 -> grab_1
#   grab_3 -> grab_2
#   grab_4 -> grab_3
for source_name, target_name in ALIGN_PAIRS:
    print("\n################################")
    print("Align pair:", source_name, "->", target_name)
    print("################################")

    # The target scan must already be aligned.
    # Otherwise, the chain order is wrong.
    if target_name not in aligned_results:
        raise RuntimeError(
            f"{target_name} is not aligned yet. "
            "Check ALIGN_PAIRS order."
        )

    # Use the already aligned version of the target scan.
    # Use the original raw version of the source scan before applying its transform.
    target_result = aligned_results[target_name]
    source_result = all_results_raw[source_name]

    # Each pair gets its own saved transform file.
    # This prevents having to manually click points again every time.
    transform_file = os.path.join(
        "output/transforms",
        f"transform_{source_name}_to_{target_name}_side_by_side_image.pickle"
    )

    # If a transform already exists and I am not forcing new clicks,
    # load the saved transform instead of clicking again.
    if os.path.exists(transform_file) and not FORCE_NEW_CLICKS:
        print("Loading transform:", transform_file)

        with open(transform_file, "rb") as f:
            data = pickle.load(f)

        R = data["R"]
        t = data["t"]

    else:
        # Manually click corresponding points between the target image and source image.
        # The function uses those clicked points to estimate a rigid transform.
        R, t = manual_alignment_from_clicks(
            target_result,
            source_result,
            target_name,
            source_name
        )

        # Optionally refine the manual alignment using ICP.
        # ICP can improve small alignment errors if the manual alignment is already close.
        # If the initial alignment is bad, ICP can sometimes make the result worse.
        if USE_ICP_REFINEMENT:
            print("\nICP refinement:")

            R, t = icp_refine_transform(
                source_result["pts3"],
                target_result["pts3"],
                R,
                t,
                max_iters=ICP_ITERS,
                max_dist=ICP_MAX_DIST,
                sample=ICP_SAMPLE
            )

        # Save the transform so future runs can reuse it.
        with open(transform_file, "wb") as f:
            pickle.dump(
                {
                    "R": R,
                    "t": t,
                    "source": source_name,
                    "target": target_name,
                    "click_mode": "side_by_side_image",
                },
                f
            )

        print("Saved transform:", transform_file)

    # Apply the estimated rigid transform to the source scan points.
    source_aligned_pts = apply_alignment(source_result["pts3"], R, t)

    # Create a new result object with the aligned 3D points.
    # Colors, faces, and other metadata are copied from the original source result.
    aligned_results[source_name] = copy_result_with_new_points(
        source_result,
        source_aligned_pts
    )

    # Save the transform in memory as well.
    pair_transforms[(source_name, target_name)] = (R, t)

    # Save the aligned source scan as a PLY file for checking in MeshLab.
    save_as_ply(
        os.path.join("output/ply", f"{source_name}_aligned.ply"),
        aligned_results[source_name]["pts3"],
        colors=aligned_results[source_name]["colors"],
        faces=aligned_results[source_name].get("faces", None)
    )

    print("Saved aligned scan:", source_name)


# Print a summary of which scans were successfully aligned.
print("\nAligned scans:")
for name in scan_names:
    if name in aligned_results:
        print(name, aligned_results[name]["pts3"].shape[1], "points")
    else:
        print(name, "not aligned")


In [ ]:
# Visualize aligned scans

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

for scan_name in scan_names:
    if scan_name not in aligned_results:
        continue

    result = aligned_results[scan_name]
    pts = result["pts3"]

    step = max(1, pts.shape[1] // 25000)

    ax.scatter(
        pts[0, ::step],
        pts[1, ::step],
        pts[2, ::step],
        s=1,
        label=scan_name
    )

ax.set_title("Aligned scans check")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.legend()
set_axes_equal(ax)

plt.show()

print("Cell 14 complete.")

Cell 14 complete.


In [ ]:
# Export each aligned scan separately before final merge

single_aligned_dir = "output/single_aligned_before_merge"
os.makedirs(single_aligned_dir, exist_ok=True)

for scan_name in scan_names:
    if scan_name not in aligned_results:
        print("Skipping, not aligned:", scan_name)
        continue

    result = aligned_results[scan_name]

    out_path = os.path.join(
        single_aligned_dir,
        f"{scan_name}_aligned_points_only.ply"
    )

    save_as_ply(
        out_path,
        result["pts3"],
        colors=result["colors"],
        faces=None
    )

    print("Saved:", out_path, "points:", result["pts3"].shape[1])


In [ ]:
# Cell 14.6: Export each aligned scan separately with extra cleanup

single_clean_dir = "output/single_aligned_clean_before_merge"
os.makedirs(single_clean_dir, exist_ok=True)

for scan_name in scan_names:
    if scan_name not in aligned_results:
        print("Skipping, not aligned:", scan_name)
        continue

    print("\nCleaning single aligned scan:", scan_name)

    result = aligned_results[scan_name]

    clean_result = remove_nonfinite_points(result)
    clean_result = remove_percentile_outliers(clean_result, low=0.5, high=99.5)

    clean_result = keep_largest_component(
        clean_result,
        radius=SINGLE_COMPONENT_RADIUS,
        min_component_size=SINGLE_MIN_COMPONENT_SIZE
    )

    out_path = os.path.join(
        single_clean_dir,
        f"{scan_name}_aligned_clean_points_only.ply"
    )

    save_as_ply(
        out_path,
        clean_result["pts3"],
        colors=clean_result["colors"],
        faces=None
    )

    print("Saved:", out_path, "points:", clean_result["pts3"].shape[1])


In [ ]:

all_pts = []
all_colors = []
all_faces = []

offset = 0

for scan_name in scan_names:
    if scan_name not in aligned_results:
        continue

    result = aligned_results[scan_name]

    pts = result["pts3"]
    colors = result["colors"]
    faces = result.get("faces", None)

    all_pts.append(pts)
    all_colors.append(colors)

    if faces is not None and len(faces) > 0:
        faces = np.asarray(faces)

        if faces.shape[0] == 3 and faces.shape[1] != 3:
            faces = faces.T

        all_faces.append(faces + offset)

    offset += pts.shape[1]

merged_pts = np.hstack(all_pts)
merged_colors = np.hstack(all_colors)
merged_faces = np.vstack(all_faces) if len(all_faces) > 0 else None

merged_result = {
    "scan_name": "merged",
    "scan_dir": DATA_DIR,
    "pts3": merged_pts,
    "colors": merged_colors,
    "faces": merged_faces,
    "pts2L": np.zeros((2, merged_pts.shape[1])),
    "pts2R": np.zeros((2, merged_pts.shape[1])),
    "matchL": np.arange(merged_pts.shape[1]),
    "matchR": np.arange(merged_pts.shape[1]),
}

print("Merged points:", merged_pts.shape[1])
print("Merged faces:", 0 if merged_faces is None else len(merged_faces))

save_as_ply(
    "output/ply/check_merged_before_final_cleanup.ply",
    merged_result["pts3"],
    colors=merged_result["colors"],
    faces=None
)

print("Cell 15 complete.")

In [ ]:
# Cell 15.5: Strong statistical outlier removal before final cleanup

def statistical_outlier_removal_result(result, k=16, std_ratio=1.2):
    pts = result["pts3"].T

    print("Statistical outlier removal input:", pts.shape[0])

    tree = cKDTree(pts)

    dists, idx = tree.query(pts, k=k + 1)

    mean_dists = np.mean(dists[:, 1:], axis=1)

    threshold = np.mean(mean_dists) + std_ratio * np.std(mean_dists)

    keep = mean_dists < threshold

    print("mean distance threshold:", threshold)
    print("kept:", np.sum(keep), "/", len(keep))

    return filter_result_by_mask(result, keep)


merged_result = statistical_outlier_removal_result(
    merged_result,
    k=16,
    std_ratio=1.0
)

save_as_ply(
    "output/ply/check_merged_after_statistical_outlier_removal.ply",
    merged_result["pts3"],
    colors=merged_result["colors"],
    faces=None
)

print("Saved output/ply/check_merged_after_statistical_outlier_removal.ply")
print("Cell 15.5 complete.")

In [ ]:
# Cell 15.7: Manual 3D bounds crop before final cleanup

def crop_result_by_bounds(result, bounds):
    pts = result["pts3"]
    keep = np.ones(pts.shape[1], dtype=bool)

    if "x" in bounds:
        keep &= (pts[0] >= bounds["x"][0]) & (pts[0] <= bounds["x"][1])

    if "y" in bounds:
        keep &= (pts[1] >= bounds["y"][0]) & (pts[1] <= bounds["y"][1])

    if "z" in bounds:
        keep &= (pts[2] >= bounds["z"][0]) & (pts[2] <= bounds["z"][1])

    print("crop kept:", np.sum(keep), "/", len(keep))

    return filter_result_by_mask(result, keep)


# 先看 Cell 18 输出的 X/Y/Z range，再慢慢改这里
bounds = {
    "x": (-80, 80),
    "y": (-90, 90),
    "z": (-20, 160),
}

merged_result = crop_result_by_bounds(merged_result, bounds)

save_as_ply(
    "output/ply/check_merged_after_bounds_crop.ply",
    merged_result["pts3"],
    colors=merged_result["colors"],
    faces=None
)

print("Saved output/ply/check_merged_after_bounds_crop.ply")

In [ ]:
# Cell 16: Final cleanup and save

final_result = remove_nonfinite_points(merged_result)
final_result = remove_percentile_outliers(final_result, low=0.2, high=99.8)

final_result = keep_largest_component(
    final_result,
    radius=FINAL_COMPONENT_RADIUS,
    min_component_size=FINAL_MIN_COMPONENT_SIZE
)

save_as_ply(
    "output/model.ply",
    final_result["pts3"],
    colors=final_result["colors"],
    faces=final_result.get("faces", None)
)

save_as_ply(
    "output/model_points_only.ply",
    final_result["pts3"],
    colors=final_result["colors"],
    faces=None
)

print("Final points:", final_result["pts3"].shape[1])
print("Final faces:", 0 if final_result.get("faces", None) is None else len(final_result["faces"]))
print("Saved output/model.ply")
print("Saved output/model_points_only.ply")
print("Cell 16 complete.")

In [ ]:
# Final Cell: Uniformly sample 90k points from final mesh surface
# This is better than voxel fuse when the point cloud is too sparse or uneven.

import os
import numpy as np

os.makedirs("output", exist_ok=True)


def sample_points_from_mesh_surface(result, target_points=90000, max_edge_length=None):
    """
    Uniformly sample points from the triangle mesh surface.

    This produces a much more uniform point cloud than voxel fusion.
    It is useful before MeshLab normal estimation and Poisson reconstruction.

    result must contain:
        result["pts3"]   shape (3, N)
        result["colors"] shape (3, N)
        result["faces"]  list/array of triangles with vertex indices
    """

    pts = result["pts3"].T
    colors = result["colors"].T
    faces = result.get("faces", None)

    if faces is None:
        raise RuntimeError(
            "final_result has no faces. "
            "Use the merged mesh result before setting faces=None."
        )

    faces = np.asarray(faces, dtype=np.int64)

    if faces.shape[0] == 0:
        raise RuntimeError("No faces available for surface sampling.")

    v0 = pts[faces[:, 0]]
    v1 = pts[faces[:, 1]]
    v2 = pts[faces[:, 2]]

    # Remove invalid triangles
    finite_mask = (
        np.all(np.isfinite(v0), axis=1)
        & np.all(np.isfinite(v1), axis=1)
        & np.all(np.isfinite(v2), axis=1)
    )

    faces = faces[finite_mask]
    v0 = v0[finite_mask]
    v1 = v1[finite_mask]
    v2 = v2[finite_mask]

    # Optional: remove triangles with very long edges
    if max_edge_length is not None:
        e01 = np.linalg.norm(v1 - v0, axis=1)
        e12 = np.linalg.norm(v2 - v1, axis=1)
        e20 = np.linalg.norm(v0 - v2, axis=1)

        edge_mask = (
            (e01 < max_edge_length)
            & (e12 < max_edge_length)
            & (e20 < max_edge_length)
        )

        faces = faces[edge_mask]
        v0 = v0[edge_mask]
        v1 = v1[edge_mask]
        v2 = v2[edge_mask]

    # Triangle areas
    cross = np.cross(v1 - v0, v2 - v0)
    areas = 0.5 * np.linalg.norm(cross, axis=1)

    area_mask = areas > 1e-10

    faces = faces[area_mask]
    v0 = v0[area_mask]
    v1 = v1[area_mask]
    v2 = v2[area_mask]
    areas = areas[area_mask]

    if faces.shape[0] == 0:
        raise RuntimeError("No valid triangles after filtering.")

    area_prob = areas / np.sum(areas)

    rng = np.random.default_rng(0)

    # Choose triangles proportional to surface area
    chosen_face_ids = rng.choice(
        faces.shape[0],
        size=target_points,
        replace=True,
        p=area_prob
    )

    chosen_faces = faces[chosen_face_ids]

    a = pts[chosen_faces[:, 0]]
    b = pts[chosen_faces[:, 1]]
    c = pts[chosen_faces[:, 2]]

    ca = colors[chosen_faces[:, 0]]
    cb = colors[chosen_faces[:, 1]]
    cc = colors[chosen_faces[:, 2]]

    # Uniform random barycentric coordinates
    r1 = rng.random(target_points)
    r2 = rng.random(target_points)

    sqrt_r1 = np.sqrt(r1)

    w0 = 1.0 - sqrt_r1
    w1 = sqrt_r1 * (1.0 - r2)
    w2 = sqrt_r1 * r2

    sampled_pts = (
        w0[:, None] * a
        + w1[:, None] * b
        + w2[:, None] * c
    )

    sampled_colors = (
        w0[:, None] * ca
        + w1[:, None] * cb
        + w2[:, None] * cc
    )

    sampled_result = result.copy()
    sampled_result["pts3"] = sampled_pts.T
    sampled_result["colors"] = sampled_colors.T
    sampled_result["faces"] = None

    print("Uniform mesh surface sampling")
    print("Input vertices :", pts.shape[0])
    print("Input faces    :", faces.shape[0])
    print("Output points  :", sampled_pts.shape[0])
    print("Target points  :", target_points)

    return sampled_result


# Try 90k and 120k.
# 90k is usually good for MeshLab normals + Poisson.
for n in [90000, 120000]:
    sampled_result = sample_points_from_mesh_surface(
        final_result,
        target_points=n,
        max_edge_length=4.0
    )

    out_path = f"output/model_surface_sampled_{n}_points_only.ply"

    save_as_ply(
        out_path,
        sampled_result["pts3"],
        colors=sampled_result["colors"],
        faces=None
    )

    print("Saved:", out_path)

In [ ]:
# Voxel fuse final merged point cloud to target around 80k-100k points

import os
import numpy as np

os.makedirs("output", exist_ok=True)


def voxel_fuse_final_result(result, cell_size=0.10):
    """
    Voxel fuse a point cloud to make point distribution more uniform.

    Smaller cell_size  -> more points, more detail
    Larger cell_size   -> fewer points, smoother / less dense

    This returns a points-only result because voxel averaging changes vertices,
    so the old faces are no longer valid.
    """

    pts = result["pts3"].T
    colors = result["colors"].T

    if pts.shape[0] == 0:
        raise RuntimeError("No points to voxel fuse.")

    # Compute voxel index for each point
    voxel_keys = np.floor(pts / cell_size).astype(np.int64)

    # Efficient grouping by voxel
    unique_keys, inverse = np.unique(voxel_keys, axis=0, return_inverse=True)

    num_voxels = unique_keys.shape[0]

    fused_pts = np.zeros((num_voxels, 3), dtype=np.float64)
    fused_colors = np.zeros((num_voxels, 3), dtype=np.float64)
    counts = np.zeros(num_voxels, dtype=np.int64)

    for i in range(pts.shape[0]):
        k = inverse[i]
        fused_pts[k] += pts[i]
        fused_colors[k] += colors[i]
        counts[k] += 1

    fused_pts = fused_pts / counts[:, None]
    fused_colors = fused_colors / counts[:, None]

    result2 = result.copy()
    result2["pts3"] = fused_pts.T
    result2["colors"] = fused_colors.T
    result2["faces"] = None

    print("Final voxel fuse")
    print("Input points :", pts.shape[0])
    print("Output points:", fused_pts.shape[0])
    print("cell_size    :", cell_size)

    return result2


# Try smaller voxel sizes.
# Your old [0.25, 0.30, 0.35...] was too large and reduced the cloud too much.
cell_sizes = [
    0.04,
    0.05,
    0.06,
    0.07,
    0.08,
    0.09,
    0.10,
    0.11,
    0.12,
    0.13,
    0.14,
    0.15,
    0.16,
    0.18,
    0.20,
    0.22,
    0.25,
]

target_points = 90000

results = []

for cs in cell_sizes:
    fused_result = voxel_fuse_final_result(final_result, cell_size=cs)

    n_points = fused_result["pts3"].shape[1]

    out_path = f"output/model_final_voxel_{cs:.2f}_points_only.ply"

    save_as_ply(
        out_path,
        fused_result["pts3"],
        colors=fused_result["colors"],
        faces=None
    )

    print("Saved:", out_path)
    print()

    results.append(
        {
            "cell_size": cs,
            "n_points": n_points,
            "path": out_path,
            "result": fused_result,
        }
    )


# Pick the version closest to target_points
best = min(results, key=lambda r: abs(r["n_points"] - target_points))

print("======================================")
print("Best voxel result")
print("target points:", target_points)
print("cell_size    :", best["cell_size"])
print("points       :", best["n_points"])
print("path         :", best["path"])
print("======================================")


# Save a convenient best-name copy
best_out_path = "output/model_final_voxel_best_points_only.ply"

save_as_ply(
    best_out_path,
    best["result"]["pts3"],
    colors=best["result"]["colors"],
    faces=None
)

print("Saved best result:", best_out_path)